# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List, Dict, Any, Tuple, Callable

from fasthtml.common import APIRouter

from cjm_fasthtml_card_stack.core.models import CardStackUrls

from cjm_fasthtml_workflow_transcript_decomp.routes.models import (
    DecompUrls, SelectionUrls, AlignmentUrls,
)

# Import core router assembly
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import init_core_routers

# Import selection router assembly
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.init import init_selection_routers

# Import decomposition router assembly
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.init import init_decomposition_routers

# Import Phase 2 (alignment) handlers — card stack UI operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.card_stack import (
    _handle_align_navigate,
    _handle_align_update_viewport,
    _handle_align_save_width,
)

# Import Phase 2 (alignment) handlers — workflow-specific operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.handlers import (
    _handle_align_init,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Alignment Router

Phase 2 right column routes for VAD audio alignment.

In [ ]:
#| export
def init_alignment_router(
    workflow: "StructureDecompWorkflow",  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/align")
    audio_src_url: str,  # URL for audio_src route (from core router)
) -> Tuple[APIRouter, AlignmentUrls, Dict[str, Callable]]:  # (router, urls, route_dict)
    """Initialize Phase 2 alignment routes."""
    router = APIRouter(prefix=prefix)

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _handle_align_init(workflow, request, sess, urls=urls)

    # -------------------------------------------------------------------------
    # Navigation
    # -------------------------------------------------------------------------

    @router
    def nav_up(request, sess):
        """Navigate to previous VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="up", urls=urls)

    @router
    def nav_down(request, sess):
        """Navigate to next VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="down", urls=urls)

    @router
    def nav_first(request, sess):
        """Navigate to first VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="first", urls=urls)

    @router
    def nav_last(request, sess):
        """Navigate to last VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="last", urls=urls)

    @router
    def nav_page_up(request, sess):
        """Navigate up by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_up", urls=urls)

    @router
    def nav_page_down(request, sess):
        """Navigate down by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_down", urls=urls)

    # -------------------------------------------------------------------------
    # Viewport and Width
    # -------------------------------------------------------------------------

    @router
    async def update_viewport(request, sess, visible_count: int):
        """Update alignment viewport with new card count."""
        return await _handle_align_update_viewport(
            workflow, request, sess, visible_count, urls=urls,
        )

    @router
    def save_width(request, sess, card_width: int):
        """Save alignment card stack width to server state."""
        return _handle_align_save_width(workflow, sess, card_width)

    # -------------------------------------------------------------------------
    # URL Bundle and Route Dict
    # -------------------------------------------------------------------------

    urls = AlignmentUrls(
        card_stack=CardStackUrls(
            nav_up=nav_up.to(),
            nav_down=nav_down.to(),
            nav_first=nav_first.to(),
            nav_last=nav_last.to(),
            nav_page_up=nav_page_up.to(),
            nav_page_down=nav_page_down.to(),
            update_viewport=update_viewport.to(),
            save_width=save_width.to(),
        ),
        init=init.to(),
        audio_src=audio_src_url,
    )

    routes = {
        "init": init,
        "nav_up": nav_up,
        "nav_down": nav_down,
        "nav_first": nav_first,
        "nav_last": nav_last,
        "nav_page_up": nav_page_up,
        "nav_page_down": nav_page_down,
        "update_viewport": update_viewport,
        "save_width": save_width,
    }

    return router, urls, routes

## Router Assembly

The `init_routers` function calls all focused router initializers and wires up cross-router dependencies.

In [ ]:
#| export
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    """Initialize and return all workflow routers."""
    base_prefix = workflow.config.route_prefix

    # Initialize focused routers
    core_routers, core_routes = init_core_routers(
        workflow, f"{base_prefix}/core"
    )
    selection_routers, selection_urls, selection_routes = init_selection_routers(
        workflow, f"{base_prefix}/selection"
    )
    decomp_routers, decomp_urls, decomp_routes = init_decomposition_routers(
        workflow, f"{base_prefix}/decomp"
    )
    align_router, align_urls, align_routes = init_alignment_router(
        workflow, f"{base_prefix}/align",
        audio_src_url=core_routes["audio_src"].to(),
    )

    # Store URL bundles on workflow for renderer access
    workflow._selection_urls = selection_urls
    workflow._decomp_urls = decomp_urls
    workflow._align_urls = align_urls
    workflow._switch_chrome_url = core_routes["switch_chrome"].to()

    # Store route dicts on workflow
    workflow._core_routes = core_routes
    workflow._selection_routes = selection_routes
    workflow._decomposition_routes = decomp_routes
    workflow._alignment_routes = align_routes

    return core_routers + selection_routers + decomp_routers + [align_router]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()